In [ ]:
!pip install transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!unzip '*.zip'

In [ ]:
import json

with open("f_messages.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dialogues = []   # 각 conversation 단위 대화 모음
labels = []      # 각 대화에 대한 라벨 (0/1)

for convo in data:
    convo_id = convo["conversation_id"]
    messages = convo["messages"]

    # 모든 메시지를 'Speaker: text' 형태로 묶기
    convo_text = " [SEP] ".join([f"{msg['author']}: {msg['text']}" for msg in messages])

    # 그루밍 의도가 하나라도 있으면 1, 아니면 0
    label = 1 if any(msg["is_offender"] for msg in messages) else 0

    dialogues.append(convo_text)
    labels.append(label)

# 출력 예시
print("Number of conversations:", len(dialogues))
print("Sample conversation:\n", dialogues[0])
print("Label:", labels[0])

In [ ]:
import pandas as pd

df = pd.DataFrame({"text": dialogues, "label": labels})
df.head()

In [ ]:
converted_texts = []

for text in df['text']:
    utterances = text.split(" [SEP] ")
    user_ids = []
    for utt in utterances:
        if ": " in utt:
            user_id = utt.split(": ")[0]
            user_ids.append(user_id)
    unique_users = list(dict.fromkeys(user_ids))
    user_map = {unique_users[0]: 0, unique_users[1]: 1}

    converted_utterances = []
    for utt in utterances:
        if ": " in utt:
            user_id, utterance_text = utt.split(": ", 1)
            new_user_id = str(user_map[user_id])
            converted_utterances.append(f"{new_user_id}: {utterance_text}")
        else:
            converted_utterances.append(utt)

    converted_text = " [SEP] ".join(converted_utterances)
    converted_texts.append(converted_text)


In [ ]:
print(converted_texts)

In [ ]:
df = pd.DataFrame({"text": converted_texts, "label": labels})
df.head()

In [ ]:
df.to_csv("labeling.csv", index=False)

In [ ]:
!pip install transformers datasets scikit-learn

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. 데이터 로딩
df = pd.read_csv("your_data.csv")  # text, label 열 필요

# 2. tokenizer 불러오기
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 3. Sliding window 적용 함수
def tokenize_and_chunk(text, label, tokenizer, chunk_size=512, stride=256):
    inputs = tokenizer(text, truncation=False, return_tensors='pt', padding=False)
    input_ids = inputs['input_ids'][0]
    attention_mask = inputs['attention_mask'][0]

    chunks = []
    for i in range(0, len(input_ids), stride):
        input_chunk = input_ids[i:i+chunk_size]
        attn_chunk = attention_mask[i:i+chunk_size]

        if len(input_chunk) < 10:  # 너무 짧은 chunk는 무시
            continue

        chunks.append({
            'input_ids': input_chunk,
            'attention_mask': attn_chunk,
            'label': label
        })

        if i + chunk_size >= len(input_ids):
            break

    return chunks

# 4. 전체 데이터에 대해 토큰화 및 chunking
all_chunks = []
for _, row in df.iterrows():
    chunks = tokenize_and_chunk(row['text'], row['label'], tokenizer)
    all_chunks.extend(chunks)

# 5. PyTorch Dataset 정의
class GroomingDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        return {
            'input_ids': chunk['input_ids'],
            'attention_mask': chunk['attention_mask'],
            'labels': torch.tensor(chunk['label'], dtype=torch.long)
        }

# 6. 데이터셋 분할
train_chunks, val_chunks = train_test_split(all_chunks, test_size=0.2, random_state=42)
train_dataset = GroomingDataset(train_chunks)
val_dataset = GroomingDataset(val_chunks)

# 7. 모델 불러오기
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# 8. Trainer 설정
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# 9. 학습 시작
trainer.train()
